In [1]:
123

123

In [1]:
import glob
import json
import random
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm


In [14]:
DATA_DIR     = './new_data'
OUTPUT_JSONL = './dssm_attn_logq_3.jsonl'

DIM          = 128
MAX_LEN      = 15
N_NEGATIVES  = 100
TEMPERATURE  = 0.1
LR           = 1e-4
WEIGHT_DECAY = 1e-4
BATCH_SIZE   = 512
EPOCHS       = 30
NEIGHBOR_K   = 200
POP_WEIGHT   = 0.07
SEED         = 42
DEVICE       = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device: {DEVICE}')


device: cuda


In [3]:
train_data = pd.concat([
    pd.read_json(path, lines=True)[['user', 'track', 'timestamp', 'time']]
    for path in glob.glob('./new_data/botify-recommender-*/data.json*')
])
train_data = train_data.drop_duplicates(subset=['user', 'track'])
print(f'Deduplicated interactions: {len(train_data):,}')


Deduplicated interactions: 214,608


In [4]:
NUM_ITEMS = int(train_data['track'].max()) + 1
PAD_IDX   = NUM_ITEMS
print(f'num_items: {NUM_ITEMS:,}')


num_items: 16,198


In [5]:
def build_sessions(data_dir, min_len=2):
    by_user = defaultdict(list)
    for path in sorted(glob.glob(f'{data_dir}/botify-recommender-*/data.json*')):
        with open(path) as f:
            for line in f:
                try:
                    ev = json.loads(line)
                except json.JSONDecodeError:
                    continue
                if ev.get('message') in ('next', 'last'):
                    by_user[int(ev['user'])].append(ev)

    sessions = []
    for evs in by_user.values():
        evs.sort(key=lambda e: int(e['timestamp']))
        buf = []
        for ev in evs:
            buf.append((int(ev['track']), float(ev.get('time', 0.0))))
            if ev['message'] == 'last':
                if len(buf) >= min_len:
                    sessions.append(buf)
                buf = []
        if len(buf) >= min_len:
            sessions.append(buf)
    return sessions


sessions = build_sessions(DATA_DIR)
print(f'Sessions: {len(sessions):,}  avg_len={np.mean([len(s) for s in sessions]):.1f}')


Sessions: 20,022  avg_len=12.1


In [6]:
def compute_item_probs(sessions, num_items):
    counts = np.zeros(num_items, dtype=np.float64)
    for sess in sessions:
        for track, _ in sess:
            if 0 <= track < num_items:
                counts[track] += 1.0
    return (counts / counts.sum().clip(1)).astype(np.float32)


def build_popular(sessions, num_items):
    score = np.zeros(num_items, dtype=np.float64)
    for sess in sessions:
        for track, t in sess:
            if 0 <= track < num_items:
                score[track] += 1.0 + max(0.0, float(t))
    return sorted(range(num_items), key=lambda i: score[i], reverse=True)


item_probs = compute_item_probs(sessions, NUM_ITEMS)
popular    = build_popular(sessions, NUM_ITEMS)
print(f'Most popular track: {popular[0]}  p={item_probs[popular[0]]:.5f}')


Most popular track: 1454  p=0.00203


In [7]:
def make_examples(sessions, num_items, popular, max_len, n_neg, seed):
    rng = random.Random(seed)
    popular_head = popular[:1024]
    examples = []

    for sess in sessions:
        tracks = [t for t, _ in sess]
        times  = [tm for _, tm in sess]

        for end in range(1, len(tracks)):
            ctx   = tracks[max(0, end - max_len):end]
            ctx_t = times[max(0, end - max_len):end]
            tgt   = tracks[end]
            banned = set(ctx) | {tgt}

            negs, tries = [], 0
            while len(negs) < n_neg and tries < n_neg * 15:
                c = rng.choice(popular_head); tries += 1
                if c not in banned and c not in negs:
                    negs.append(c)
            while len(negs) < n_neg:
                c = rng.randrange(num_items)
                if c not in banned and c not in negs:
                    negs.append(c)

            weight = 1.0 + 2.0 * max(0.0, min(float(times[end]), 1.0))
            examples.append((ctx, ctx_t, tgt, negs, weight))

    return examples


random.seed(SEED)
examples = make_examples(sessions, NUM_ITEMS, popular, MAX_LEN, N_NEGATIVES, SEED)
print(f'Training examples: {len(examples):,}')


Training examples: 222,621


In [8]:
class SessionDataset(Dataset):
    def __init__(self, examples, max_len, pad_idx):
        self.data    = examples
        self.max_len = max_len
        self.pad_idx = pad_idx

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        ctx, ctx_t, tgt, negs, w = self.data[idx]
        ctx   = ctx[-self.max_len:]
        ctx_t = ctx_t[-self.max_len:]
        pad   = self.max_len - len(ctx)
        ctx_p   = [self.pad_idx] * pad + ctx
        ctx_t_p = [0.0] * pad + [max(0.0, min(t, 1.0)) for t in ctx_t]
        return (
            torch.tensor(ctx_p,   dtype=torch.long),
            torch.tensor(ctx_t_p, dtype=torch.float32),
            torch.tensor(tgt,     dtype=torch.long),
            torch.tensor(negs,    dtype=torch.long),
            torch.tensor(w,       dtype=torch.float32),
        )


loader = DataLoader(
    SessionDataset(examples, MAX_LEN, PAD_IDX),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=(DEVICE == 'cuda'),
)
print(f'Batches/epoch: {len(loader):,}')


Batches/epoch: 435


In [9]:
class DSSM_Attn_LogQ_Rec(nn.Module):
    def __init__(self, num_items, dim, pad_idx):
        super().__init__()
        self.pad_idx   = pad_idx
        self.user_emb  = nn.Embedding(num_items + 1, dim, padding_idx=pad_idx)
        self.item_emb  = nn.Embedding(num_items, dim)
        self.decay     = nn.Parameter(torch.tensor(1.5))
        self.attn_temp = nn.Parameter(torch.tensor(0.3))
        nn.init.normal_(self.user_emb.weight, std=0.01)
        nn.init.normal_(self.item_emb.weight, std=0.01)
        with torch.no_grad():
            self.user_emb.weight[pad_idx].zero_()

    def encode_user(self, ctx, ctx_t):
        B, L   = ctx.shape
        mask   = ctx.ne(self.pad_idx)
        pos    = torch.arange(L, device=ctx.device).float()
        lam    = self.decay.abs().clamp(0.05, 2.0)
        rec    = torch.exp(-lam * (L - 1 - pos))
        a      = rec.unsqueeze(0) * ctx_t.clamp(1e-3, 1.0)
        a      = a / self.attn_temp.abs().clamp(1e-2)
        a      = a.masked_fill(~mask, float('-inf'))
        w      = torch.softmax(a, dim=1).unsqueeze(-1)
        return F.normalize((self.user_emb(ctx) * w).sum(1), dim=1)

    def encode_item(self, ids):
        return F.normalize(self.item_emb(ids), dim=-1)

    def forward(self, ctx, ctx_t, pos_ids, neg_ids, log_q, weights, temp):
        u   = self.encode_user(ctx, ctx_t)
        pv  = self.encode_item(pos_ids)
        nv  = self.encode_item(neg_ids)

        ps  = (u * pv).sum(1) / temp - log_q[pos_ids]
        ns  = (nv * u.unsqueeze(1)).sum(2) / temp - log_q[neg_ids]

        loss = -F.logsigmoid(ps.unsqueeze(1) - ns).mean(1)
        return (loss * weights).sum() / weights.sum().clamp_min(1.0)


torch.manual_seed(SEED)
model = DSSM_Attn_LogQ_Rec(NUM_ITEMS, DIM, PAD_IDX).to(DEVICE)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')


Parameters: 4,146,818


In [10]:
log_q = torch.tensor(np.log(item_probs.clip(1e-9)), dtype=torch.float32, device=DEVICE)

optimizer = torch.optim.AdamW([
    {'params': [model.user_emb.weight, model.item_emb.weight], 'lr': LR},
    {'params': [model.decay, model.attn_temp], 'lr': LR * 0.1},
], weight_decay=WEIGHT_DECAY)

# optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR * 0.1)

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = total_w = 0.0
    bar = tqdm(loader, desc=f'Epoch {epoch}/{EPOCHS}', leave=False)

    for ctx, ctx_t, tgt, negs, w in bar:
        ctx   = ctx.to(DEVICE)
        ctx_t = ctx_t.to(DEVICE)
        tgt   = tgt.to(DEVICE)
        negs  = negs.to(DEVICE)
        w     = w.to(DEVICE)

        loss = model(ctx, ctx_t, tgt, negs, log_q, w, TEMPERATURE)
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += float(loss.detach()) * float(w.sum())
        total_w    += float(w.sum())
        bar.set_postfix(loss=f'{total_loss / max(total_w, 1):.5f}')

    scheduler.step()
    print(
        f'Epoch {epoch:>2}/{EPOCHS}  '
        f'loss={total_loss / max(total_w, 1):.5f}  '
        f'lr={scheduler.get_last_lr()[0]:.2e}  '
        f'decay={model.decay.item():.3f}  '
        f'attn_temp={model.attn_temp.item():.3f}'
    )


Epoch 1/30:   0%|          | 0/435 [00:00<?, ?it/s]

Epoch  1/30  loss=0.25722  lr=9.98e-05  decay=1.495  attn_temp=0.306


Epoch 2/30:   0%|          | 0/435 [00:00<?, ?it/s]

Epoch  2/30  loss=0.05628  lr=9.90e-05  decay=1.492  attn_temp=0.309


Epoch 3/30:   0%|          | 0/435 [00:00<?, ?it/s]

...

Epoch 28/30  loss=0.00327  lr=1.10e-05  decay=1.419  attn_temp=0.374


Epoch 29/30:   0%|          | 0/435 [00:00<?, ?it/s]

Epoch 29/30  loss=0.00323  lr=1.02e-05  decay=1.415  attn_temp=0.378


Epoch 30/30:   0%|          | 0/435 [00:00<?, ?it/s]

Epoch 30/30  loss=0.00320  lr=1.00e-05  decay=1.411  attn_temp=0.381


In [11]:
model.eval()
with torch.no_grad():
    anchor_vecs = F.normalize(model.user_emb.weight[:-1], dim=1).cpu().numpy().astype('float32')
    item_vecs   = F.normalize(model.item_emb.weight,       dim=1).cpu().numpy().astype('float32')

pop_scores = np.log1p(item_probs * len(item_probs)).astype('float32')
pop_scores /= pop_scores.max().clip(1)

n         = anchor_vecs.shape[0]
k         = min(NEIGHBOR_K, item_vecs.shape[0] - 1)
neighbors = np.empty((n, k), dtype=np.int32)

for start in tqdm(range(0, n, 512), desc='Neighbors'):
    end    = min(start + 512, n)
    scores = anchor_vecs[start:end] @ item_vecs.T
    scores += POP_WEIGHT * pop_scores
    scores[np.arange(end - start), np.arange(start, end)] = -np.inf
    top    = np.argpartition(-scores, kth=k - 1, axis=1)[:, :k]
    top_s  = np.take_along_axis(scores, top, axis=1)
    order  = np.argsort(-top_s, axis=1)
    neighbors[start:end] = np.take_along_axis(top, order, axis=1)

print(f'neighbors: {neighbors.shape}')


Neighbors:   0%|          | 0/32 [00:00<?, ?it/s]

neighbors: (16198, 200)


In [16]:
Path(OUTPUT_JSONL).parent.mkdir(parents=True, exist_ok=True)

with open(OUTPUT_JSONL, 'w') as f:
    for item_id, row in enumerate(neighbors):
        f.write(json.dumps({'item_id': item_id, 'recommendations': row[:20].tolist()}) + '\n')

print(f'Saved {neighbors.shape[0]:,} items → {OUTPUT_JSONL}')


Saved 16,198 items → ./dssm_attn_logq_3.jsonl


In [13]:
def recall_at_k(neighbors, sessions, k=10, n_eval=2000):
    rng = random.Random(SEED)
    hits = tested = 0
    for sess in rng.sample(sessions, min(n_eval, len(sessions))):
        if len(sess) < 2:
            continue
        anchor, target = sess[-2][0], sess[-1][0]
        if anchor >= neighbors.shape[0]:
            continue
        hits   += int(target in neighbors[anchor, :k])
        tested += 1
    return hits / max(tested, 1)


for k in (10, 20, 50, 100):
    print(f'Recall@{k:>3} = {recall_at_k(neighbors, sessions, k=k):.4f}')


Recall@ 10 = 0.5095
Recall@ 20 = 0.5935
Recall@ 50 = 0.6880
Recall@100 = 0.7560
